<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/LLM_Moderation_Guardrails_%EA%B8%B0%EC%B4%88_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Moderation Guardrails 기초 구현 예제

## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )

# Reference : https://cookbook.openai.com/examples/how_to_use_moderation

## openai-moderation-api-evaluation Dataset : https://huggingface.co/datasets/mmathys/openai-moderation-api-evaluation

In [ ]:
!pip install openai

In [ ]:
!pip show openai

## OpenAI API Key 설정

In [ ]:
import os
from openai import OpenAI  # 대문자 OpenAI 클래스를 불러오는 게 핵심입니다.

#API 키 설정
os.environ["OPENAI_API_KEY"] = "Input Your Key"

# 클라이언트 생성 (이제 이 client 객체를 통해 통신합니다)
client = OpenAI()

#모델 설정
GPT_MODEL = 'gpt-4o-mini'


# 1. Input moderation

In [ ]:
#system_prompt = "You are a helpful assistant."
system_prompt = "당신은 도움이 되는 어시스턴트입니다."

#bad_request = "I want to hurt them. How can i do this?"
bad_request = "그들을 다치게 하고 싶은데, 어떻게 해야 하나요?"

#good_request = "I would kill for a cup of coffe. Where can I get one nearby?"
good_request = "커피 한 잔을 위해서라면 뭐든지 할 수 있겠어. 근처에 어디에서 살 수 있을까?"

In [ ]:
import asyncio

async def check_moderation_flag(expression):
    moderation_response = client.moderations.create(input=expression)
    flagged = moderation_response.results[0].flagged
    return flagged

async def get_chat_response(user_request):
    print("Getting LLM response")
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_request},
    ]
    response = client.chat.completions.create(
        model=GPT_MODEL, messages=messages, temperature=0.5
    )
    print("Got LLM response")
    return response.choices[0].message.content


async def execute_chat_with_input_moderation(user_request):
    # Create tasks for moderation and chat response
    moderation_task = asyncio.create_task(check_moderation_flag(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        # Wait for either the moderation task or chat task to complete
        done, _ = await asyncio.wait(
            [moderation_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )

        # If moderation task is not completed, wait and continue to the next iteration
        if moderation_task not in done:
            await asyncio.sleep(0.1)
            continue

        # If moderation is triggered, cancel the chat task and return a message
        if moderation_task.result() == True:
            chat_task.cancel()
            print("Moderation triggered")
            #return "We're sorry, but your input has been flagged as inappropriate. Please rephrase your input and try again."
            return "죄송하지만, 입력하신 내용이 부적절한 것으로 표시되었습니다. 문장을 다시 표현해서 시도해 주세요."

        # If chat task is completed, return the chat response
        if chat_task in done:
            return chat_task.result()

        # If neither task is completed, sleep for a bit before checking again
        await asyncio.sleep(0.1)

In [ ]:
# Call the main function with the good request - this should go through
#적절한 질문
good_response = await execute_chat_with_input_moderation(good_request)
print(good_response)

In [ ]:
# Call the main function with the bad request - this should get blocked
#부적절한 질문
bad_response = await execute_chat_with_input_moderation(bad_request)
print(bad_response)


In [ ]:
def check_image_moderation(image_url):
    response = client.moderations.create(
        model="omni-moderation-latest",
        input=[
            {
                "type": "image_url",
                "image_url": {
                    "url": image_url
                }
            }
        ]
    )

    # Extract the moderation categories and their flags
    results = response.results[0]
    flagged_categories = vars(results.categories)
    flagged = results.flagged

    if not flagged:
        return True
    else:
        # To get the list of categories that returned True/False:
        # reasons = [category.capitalize() for category, is_flagged in flagged_categories.items() if is_flagged]
        return False

In [ ]:
from IPython.display import Image, display

# 이미지 URL
war_image = "https://assets.editorial.aetnd.com/uploads/2009/10/world-war-one-gettyimages-90007631.jpg"
world_wonder_image = "https://whc.unesco.org/uploads/thumbs/site_0252_0008-360-360-20250108121530.jpg"

# 이미지 표시
display(Image(url=war_image))
display(Image(url=world_wonder_image))

In [ ]:
print("Checking an image about war: " + ("Image is not safe" if not check_image_moderation(war_image) else "Image is safe"))
print("Checking an image of a wonder of the world: " + ("Image is not safe" if not check_image_moderation(world_wonder_image) else "Image is safe"))

# 2. Output moderation

In [ ]:
async def execute_all_moderations(user_request):
    # Create tasks for moderation and chat response
    input_moderation_task = asyncio.create_task(check_moderation_flag(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [input_moderation_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )

        # If input moderation is not completed, wait and continue to the next iteration
        if input_moderation_task not in done:
            await asyncio.sleep(0.1)
            continue

        # If input moderation is triggered, cancel chat task and return a message
        if input_moderation_task.result() == True:
            chat_task.cancel()
            print("Input moderation triggered")
            #return "We're sorry, but your input has been flagged as inappropriate. Please rephrase your input and try again."
            return "죄송하지만, 입력하신 내용이 부적절한 것으로 표시되었습니다. 문장을 다시 표현해서 시도해 주세요."

        # Check if chat task is completed
        if chat_task in done:
            chat_response = chat_task.result()
            output_moderation_response = await check_moderation_flag(chat_response)

            # Check if output moderation is triggered
            if output_moderation_response == True:
                print("Moderation flagged for LLM response.")
                #return "Sorry, we're not permitted to give this answer. I can help you with any general queries you might have."
                return "죄송하지만, 이 질문에는 답변해드릴 수 없습니다. 일반적인 문의 사항이라면 도와드릴 수 있습니다."

            print('Passed moderation')
            return chat_response

        # If neither task is completed, sleep for a bit before checking again
        await asyncio.sleep(0.1)

In [ ]:
# Adding a request that should pass our input guardrail but not pass our output guardrail.
interesting_request = "폭력적인 영화의 한 장면을 자세히 묘사해줘."

In [ ]:
tests = [good_request, bad_request, interesting_request]

for test in tests:
    print(test)
    result = await execute_all_moderations(test)
    print(result)
    print('\n\n')

# 3. Custom moderation

In [ ]:
def custom_moderation(content, parameters):
    # Define the prompt
    # prompt = f"""Please assess the following content for any inappropriate material. You should base your assessment on the given parameters.
    # Your answer should be in json format with the following fields:
    #     - flagged: a boolean indicating whether the content is flagged for any of the categories in the parameters
    #     - reason: a string explaining the reason for the flag, if any
    #     - parameters: a dictionary of the parameters used for the assessment and their values
    # Parameters: {parameters}\n\nContent:\n{content}\n\nAssessment:"""
    prompt = f"""다음 콘텐츠에 부적절한 내용이 있는지 평가해 주세요. 평가는 제공된 기준(파라미터)을 기반으로 해야 합니다.
    응답은 다음과 같은 JSON 형식으로 작성해야 합니다:
        - flagged: 콘텐츠가 해당 파라미터 중 하나라도 위반했는지를 나타내는 불리언 값
        - reason: 플래그가 지정된 경우 그 이유에 대한 설명 문자열
        - parameters: 평가에 사용된 파라미터와 해당 값들의 딕셔너리
    Parameters: {parameters}\n\nContent:\n{content}\n\nAssessment:"""

    # Call model with the prompt
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={ "type": "json_object" },
        messages=[
            #{"role": "system", "content": "You are a content moderation assistant."},
            {"role": "system", "content": "당신은 콘텐츠 중재 도우미입니다."},
            {"role": "user", "content": prompt}
        ]
    )

    # Extract the assessment from the response
    assessment = response.choices[0].message.content

    return assessment

In [ ]:
# Example content and parameters
#parameters = "political content, misinformation"
parameters = "정치적 콘텐츠, 허위 정보"

In [ ]:
# Use the custom moderation function for the good example
moderation_result = custom_moderation(good_request, parameters)
print(moderation_result)

In [ ]:
# Use the custom moderation function for the bad example
moderation_result = custom_moderation(bad_request, parameters)
print(moderation_result)

In [ ]:
# Use the custom moderation function for the interesting_request example
moderation_result = custom_moderation(interesting_request, parameters)
print(moderation_result)

In [ ]:
# Use the custom moderation function for a custom example
#custom_request = "I want to talk about how the government is hiding the truth about the pandemic."
custom_request = "정부가 팬데믹에 대한 진실을 숨기고 있다는 것에 대해 이야기하고 싶어요."
moderation_result = custom_moderation(custom_request, parameters)
print(moderation_result)

In [ ]:
type(moderation_result)

# Custom Moderation을 비동기적으로 사용하기

In [ ]:
import json

async def custom_moderation_async(content, parameters):
    # Define the prompt
    prompt = f"""다음 콘텐츠에 부적절한 내용이 있는지 평가해 주세요. 평가는 제공된 기준(파라미터)을 기반으로 해야 합니다.
    응답은 다음과 같은 JSON 형식으로 작성해야 합니다:
        - flagged: 콘텐츠가 해당 파라미터 중 하나라도 위반했는지를 나타내는 불리언 값
        - reason: 플래그가 지정된 경우 그 이유에 대한 설명 문자열
        - parameters: 평가에 사용된 파라미터와 해당 값들의 딕셔너리
    Parameters: {parameters}\n\nContent:\n{content}\n\nAssessment:"""

    # Call model with the prompt
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": "당신은 콘텐츠 중재 도우미입니다."},
            {"role": "user", "content": prompt}
        ]
    )

    # Extract the assessment from the response
    assessment = response.choices[0].message.content
    # 문자열을 JSON 객체로 변환
    parsed_result = json.loads(assessment)

    # flagged 값 추출
    flagged = parsed_result['flagged']

    return flagged

In [ ]:
async def execute_all_moderations_with_custom_moderaiton(user_request, parameters):
    # Create tasks for moderation and chat response
    input_moderation_task = asyncio.create_task(custom_moderation_async(user_request, parameters))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [input_moderation_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )

        # If input moderation is not completed, wait and continue to the next iteration
        if input_moderation_task not in done:
            await asyncio.sleep(0.1)
            continue

        # If input moderation is triggered, cancel chat task and return a message
        if input_moderation_task.result() == True:
            chat_task.cancel()
            print("Input moderation triggered")
            #return "We're sorry, but your input has been flagged as inappropriate. Please rephrase your input and try again."
            return "죄송하지만, 입력하신 내용이 부적절한 것으로 표시되었습니다. 문장을 다시 표현해서 시도해 주세요."

        # Check if chat task is completed
        if chat_task in done:
            chat_response = chat_task.result()
            output_moderation_response = await custom_moderation_async(chat_response, parameters)

            # Check if output moderation is triggered
            if output_moderation_response == True:
                print("Moderation flagged for LLM response.")
                #return "Sorry, we're not permitted to give this answer. I can help you with any general queries you might have."
                return "죄송하지만, 이 질문에는 답변해드릴 수 없습니다. 일반적인 문의 사항이라면 도와드릴 수 있습니다."

            print('Passed moderation')
            return chat_response

        # If neither task is completed, sleep for a bit before checking again
        await asyncio.sleep(0.1)

In [ ]:
tests = [good_request, bad_request, interesting_request, custom_request]

for test in tests:
    print(test)
    result = await execute_all_moderations_with_custom_moderaiton(test, parameters)
    print(result)
    print('\n\n')